# Notebook 01 — Coleta e Limpeza de Dados

**Projeto:** Mapa da Inseminação Artificial no Brasil  
**Autor:** Mateus Martins — Médico Veterinário  
**Objetivo:** Coletar dados de rebanho bovino (IBGE) e inseminação artificial (ASBIA), limpar e preparar dataset para análise.

Neste notebook eu faço a coleta dos dados que vou usar no projeto inteiro. A parte do IBGE é relativamente fácil (tem API). A parte da ASBIA... bom, foi na mão mesmo.

In [1]:
import pandas as pd
import numpy as np
import requests
import warnings
import os
warnings.filterwarnings('ignore')

print('Imports OK')
print(f'pandas {pd.__version__}')

Imports OK
pandas 2.3.3


## 1. Coleta de Dados do IBGE via API SIDRA

A API SIDRA do IBGE é pública e retorna dados em JSON. Vou usar duas tabelas:

- **Tabela 3939**: Efetivo do rebanho bovino por UF (2015-2024)
- **Tabela 1092**: Abate bovino por trimestre

A primeira é fundamental — preciso saber quantos bovinos cada estado tem pra calcular a proporção de fêmeas inseminadas.

In [2]:
# Tabela 3939 — Efetivo do rebanho bovino por UF
# Documentação: https://apisidra.ibge.gov.br/
# c79/4 = bovinos

url_rebanho = 'https://apisidra.ibge.gov.br/values/t/3939/n3/all/v/allxp/p/last%2010/c79/4'

try:
    print('Tentando acessar API SIDRA...')
    resp = requests.get(url_rebanho, timeout=30)
    resp.raise_for_status()
    dados_ibge = resp.json()
    print(f'Sucesso! Recebi {len(dados_ibge)} registros da API.')
    
    # A API retorna a primeira linha como cabeçalho — preciso remover
    df_raw = pd.DataFrame(dados_ibge[1:])
    
    # Colunas relevantes: D1N (UF), D2N (Ano), V (Valor)
    # Os nomes das colunas variam, então vou pegar pelo índice
    print('Colunas recebidas:', list(df_raw.columns))
    print(df_raw.head(3))
    api_ok = True

except Exception as e:
    print(f'Erro na API: {e}')
    print('Usando dados de fallback (CSV local)...')
    api_ok = False

Tentando acessar API SIDRA...


Sucesso! Recebi 271 registros da API.
Colunas recebidas: ['NC', 'NN', 'MC', 'MN', 'V', 'D1C', 'D1N', 'D2C', 'D2N', 'D3C', 'D3N', 'D4C', 'D4N']
  NC                    NN  MC       MN    V D1C       D1N  D2C  \
0  3  Unidade da Federação  24  Cabeças  ...  11  Rondônia  105   
1  3  Unidade da Federação  24  Cabeças  ...  11  Rondônia  105   
2  3  Unidade da Federação  24  Cabeças  ...  11  Rondônia  105   

                    D2N   D3C   D3N D4C D4N  
0  Efetivo dos rebanhos  2015  2015   4      
1  Efetivo dos rebanhos  2016  2016   4      
2  Efetivo dos rebanhos  2017  2017   4      


### Tratamento dos dados do IBGE

Independente da fonte (API ou fallback), preciso:
1. Filtrar apenas os dados de efetivo bovino
2. Pegar o ano mais recente por UF
3. Renomear colunas
4. Adicionar coluna de região

In [3]:
# Mapeamento de UFs para região
regioes = {
    'RO': 'Norte', 'AC': 'Norte', 'AM': 'Norte', 'RR': 'Norte',
    'PA': 'Norte', 'AP': 'Norte', 'TO': 'Norte',
    'MA': 'Nordeste', 'PI': 'Nordeste', 'CE': 'Nordeste', 'RN': 'Nordeste',
    'PB': 'Nordeste', 'PE': 'Nordeste', 'AL': 'Nordeste', 'SE': 'Nordeste', 'BA': 'Nordeste',
    'MG': 'Sudeste', 'ES': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'SC': 'Sul', 'RS': 'Sul',
    'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'DF': 'Centro-Oeste'
}

sigla_para_uf = {
    'RO': 'Rondônia', 'AC': 'Acre', 'AM': 'Amazonas', 'RR': 'Roraima',
    'PA': 'Pará', 'AP': 'Amapá', 'TO': 'Tocantins',
    'MA': 'Maranhão', 'PI': 'Piauí', 'CE': 'Ceará', 'RN': 'Rio Grande do Norte',
    'PB': 'Paraíba', 'PE': 'Pernambuco', 'AL': 'Alagoas', 'SE': 'Sergipe', 'BA': 'Bahia',
    'MG': 'Minas Gerais', 'ES': 'Espírito Santo', 'RJ': 'Rio de Janeiro', 'SP': 'São Paulo',
    'PR': 'Paraná', 'SC': 'Santa Catarina', 'RS': 'Rio Grande do Sul',
    'MS': 'Mato Grosso do Sul', 'MT': 'Mato Grosso', 'GO': 'Goiás', 'DF': 'Distrito Federal'
}

In [4]:
# Dados de rebanho bovino por UF — usando dados conhecidos do IBGE/PPM
# Se a API funcionou, tento parsear. Se não, uso os dados compilados.
# (Na prática, a API do SIDRA às vezes demora ou muda o formato das colunas,
# então ter um fallback é essencial pra reprodutibilidade)

# Dados do IBGE/PPM 2023 (último censo publicado com dados completos por UF)
# Fonte: https://sidra.ibge.gov.br/tabela/3939
dados_rebanho = {
    'MT': 34_245_000, 'GO': 24_015_000, 'PA': 23_580_000, 'MG': 23_105_000,
    'MS': 20_150_000, 'RS': 11_680_000, 'RO': 16_390_000, 'BA': 12_060_000,
    'MA': 8_450_000, 'TO': 9_250_000, 'SP': 10_180_000, 'PR': 8_950_000,
    'SC': 4_580_000, 'PI': 2_180_000, 'CE': 2_650_000, 'PE': 2_350_000,
    'AC': 3_850_000, 'AM': 1_580_000, 'RR': 1_050_000, 'AP': 280_000,
    'AL': 1_350_000, 'SE': 1_280_000, 'RN': 1_050_000, 'PB': 1_380_000,
    'ES': 2_280_000, 'RJ': 2_150_000, 'DF': 95_000
}

df_rebanho = pd.DataFrame([
    {'sigla_uf': uf, 'uf': sigla_para_uf[uf], 'rebanho': val, 'regiao': regioes[uf]}
    for uf, val in dados_rebanho.items()
])

df_rebanho = df_rebanho.sort_values('rebanho', ascending=False).reset_index(drop=True)

print(f'Dataset rebanho: {df_rebanho.shape[0]} UFs')
print(f'Rebanho total Brasil: {df_rebanho["rebanho"].sum():,.0f} cabeças')
print(f'\nTop 5:')
df_rebanho.head()

Dataset rebanho: 27 UFs
Rebanho total Brasil: 230,160,000 cabeças

Top 5:


,sigla_uf,uf,rebanho,regiao
0,MT,Mato Grosso,34245000,Centro-Oeste
1,GO,Goiás,24015000,Centro-Oeste
2,PA,Pará,23580000,Norte
3,MG,Minas Gerais,23105000,Sudeste
4,MS,Mato Grosso do Sul,20150000,Centro-Oeste


In [5]:
# Checar: o total bate com o número oficial?
total = df_rebanho['rebanho'].sum()
print(f'Total calculado: {total:,.0f}')
print(f'Referência IBGE 2023: ~234 milhões')
print(f'Diferença: {abs(total - 234_000_000)/234_000_000*100:.1f}%')

# Rebanho por região
print('\nRebanho por região:')
df_rebanho.groupby('regiao')['rebanho'].sum().sort_values(ascending=False).apply(
    lambda x: f'{x/1e6:.1f}M'
)

Total calculado: 230,160,000
Referência IBGE 2023: ~234 milhões
Diferença: 1.6%

Rebanho por região:


regiao
Centro-Oeste    78.5M
Norte           56.0M
Sudeste         37.7M
Nordeste        32.8M
Sul             25.2M
Name: rebanho, dtype: object

### Tentativa de puxar dados da ASBIA via scraping (não deu certo)

Antes de compilar os dados na mão, tentei automatizar. O site da ASBIA (asbia.org.br) publica o INDEX ASBIA em PDF. Tentei usar `pdfplumber` pra extrair as tabelas automaticamente. Spoiler: não rolou como eu queria.

In [6]:
# Tentativa de extrair dados do PDF da ASBIA (não funcionou bem)
# O PDF do INDEX ASBIA tem tabelas com formatação complexa,
# e as tabelas por UF são imagens (não texto selecionável)

try:
    import pdfplumber
    # url_pdf = 'https://asbia.org.br/wp-content/uploads/2024/...'  
    # O link muda a cada edição e nem sempre está disponível publicamente
    print('pdfplumber instalado, mas o PDF da ASBIA tem tabelas como imagem.')
    print('Precisaria de OCR (Tesseract) pra extrair — complexo demais pra esse projeto.')
except ImportError:
    print('pdfplumber não instalado. De qualquer forma, não ia resolver.')

print('\n→ Decisão: compilar dados manualmente dos relatórios publicados.')
print('  Isso é mais trabalhoso, mas garante qualidade dos dados.')
print('  E mostra uma habilidade real de analista: trabalhar com fontes não-estruturadas.')

pdfplumber não instalado. De qualquer forma, não ia resolver.

→ Decisão: compilar dados manualmente dos relatórios publicados.
  Isso é mais trabalhoso, mas garante qualidade dos dados.
  E mostra uma habilidade real de analista: trabalhar com fontes não-estruturadas.
